# 第3章：张量与神经网络基础

> **预计学习时间：约 30 分钟**
>
> 本章将带你理解张量、神经网络和训练过程的核心原理。张量是数组的自然延伸，理解它是掌握深度学习的第一步。读完本章后，你将理解神经网络如何"学习"，以及 PyTorch 如何自动帮你计算梯度。

**运行环境**: Kaggle Notebook (免费 T4 GPU)
**依赖库**: PyTorch, NumPy, Matplotlib

---
## 0. 导入库与设备设置

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import time

# 设置设备：优先使用 GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

# 设置随机种子，确保结果可复现
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

print(f"PyTorch 版本: {torch.__version__}")

---
## 1. 张量基础：从标量到高维数据

### 什么是张量？

张量就是**任意维度的数组容器**。你一定用过数组和矩阵，张量只是它们的自然延伸：

| 维度 | 数学名称 | 代码示例 | 在深度学习中的用途 |
|------|---------|---------|-------------------|
| 0维 | 标量 (Scalar) | `torch.tensor(3.14)` | 损失值、学习率 |
| 1维 | 向量 (Vector) | `torch.tensor([1,2,3])` | 偏置、词向量 |
| 2维 | 矩阵 (Matrix) | `torch.tensor([[1,2],[3,4]])` | 权重矩阵、一批数据 |
| 3维 | 三维张量 | `torch.randn(2,3,4)` | 一批序列的词向量 |
| N维 | N维张量 | `torch.randn(2,8,10,64)` | 注意力机制中的张量 |

![张量维度演进](images/tensor_dimensions.png)

*图1：张量维度演进*

### 用具体例子理解每种张量

```
0维张量（标量）：一个数
  3.14          用途：损失值、学习率、准确率
                代码：torch.tensor(3.14)

1维张量（向量）：一排数
  [1, 2, 3, 4]   用途：一个词的嵌入向量、偏置参数
                  代码：torch.tensor([1, 2, 3, 4])
                  shape: (4,)

2维张量（矩阵）：一张表
  [[1, 2, 3],    用途：一批词向量、权重矩阵
   [4, 5, 6]]    代码：torch.tensor([[1,2,3],[4,5,6]])
                 shape: (2, 3)  — 2行3列

3维张量：一摞表
  shape: (batch, seq_len, embed_dim)
  意思：几个句子 x 每句几个词 x 每个词的向量
  shape: (2, 2, 3) → 2批 x 每批2行 x 每行3个数
```

### 类比：用图片理解张量维度

```
一个像素的颜色：
  RGB(255, 128, 0)     ← 3个数 = 向量

一张图片的所有像素：
  width x height x 3   ← 3维张量！
  shape: (高度, 宽度, 3个颜色通道)

一批图片：
  [image1, image2, image3, ...]
  shape: (图片数, 高度, 宽度, 通道数) ← 4维张量！
```

### 张量的核心属性：shape

shape 是张量最重要的属性，它告诉你数据的"形状"：

| 场景 | shape | 含义 |
|------|-------|------|
| 一个损失值 | `()` | 标量 |
| 一个词向量 | `(768,)` | 768维向量 |
| 一个权重矩阵 | `(768, 768)` | 768x768 |
| 一批句子的词向量 | `(32, 128, 768)` | 32个句子, 每句128个词, 每个词768维 |

**GPT 的输入就是这个形状：`(batch_size, sequence_length, hidden_size)`**

> shape 决定了张量里有多少数据。元素总量 = shape 中所有维度相乘。
> `(32, 128, 768)` → 32 x 128 x 768 = 3,145,728 个数字

### 张量的运算

```
1. 逐元素运算（对应位置直接计算）：

   [1, 2, 3]  +  [4, 5, 6]  =  [5, 7, 9]
   [1, 2, 3]  *  [4, 5, 6]  =  [4, 10, 18]

2. 矩阵乘法（神经网络的核心运算）：

   [1, 2]     [5, 6]     [1x5+2x7, 1x6+2x8]     [19, 22]
   [3, 4]  x  [7, 8]  =  [3x5+4x7, 3x6+4x8]  =  [43, 50]

   这就是神经网络"思考"的方式！输入 x 权重 = 输出

3. 广播（Broadcasting，自动扩展维度）：

   [1, 2, 3]  +  10  =  [11, 12, 13]
   10 自动扩展成 [10, 10, 10] 再相加
```

> 💡 **记忆要点**
> - 张量 = 任意维度的数组，0维是标量，1维是向量，2维是矩阵，3维以上是高维张量
> - **shape** 是最重要的属性，决定了数据的形状和大小
> - 大模型的输入通常是 3维张量：**(batch_size, seq_len, hidden_dim)**
> - 矩阵乘法是神经网络最核心的运算，GPU 就是为了加速这个运算

**动手体验**：让我们创建各种张量，感受它们的形状和运算。

In [ ]:
# =============================================
# 1.1 创建张量的多种方式
# =============================================

# --- 从 Python 列表创建 ---
t1 = torch.tensor([1, 2, 3, 4])
print(f"从列表创建: {t1}")
print(f"  形状: {t1.shape}")
print(f"  数据类型: {t1.dtype}")
print(f"  维度数: {t1.dim()}")

# --- 从 NumPy 数组创建 ---
np_array = np.array([[1.0, 2.0], [3.0, 4.0]])
t2 = torch.from_numpy(np_array)
print(f"\n从 NumPy 创建: {t2}")
print(f"  形状: {t2.shape}  (2行2列的矩阵)")

# --- 创建特殊张量 ---
zeros = torch.zeros(3, 4)          # 全零，常用于初始化
ones = torch.ones(3, 4)            # 全一
rand = torch.randn(3, 4)           # 标准正态分布随机数，最常用
arange = torch.arange(0, 10, 2)    # 等差序列
eye = torch.eye(3)                 # 单位矩阵

print(f"\n全零张量 (3x4):\n{zeros}")
print(f"\n随机张量 (3x4):\n{rand}")
print(f"\n等差序列: {arange}")
print(f"\n单位矩阵:\n{eye}")

In [ ]:
# =============================================
# 1.2 张量的数据类型
# =============================================
# 数据类型决定了精度和内存占用

print("常见数据类型及其内存占用：")
print("=" * 55)

dtypes = {
    "float32 (默认)": torch.float32,
    "float16 (半精度)": torch.float16,
    "bfloat16 (脑浮点)": torch.bfloat16,
    "int64 (默认整数)": torch.int64,
    "int32": torch.int32,
    "bool": torch.bool,
}

for name, dtype in dtypes.items():
    t = torch.tensor([1.0], dtype=dtype) if dtype != torch.bool else torch.tensor([True])
    bytes_per_elem = t.element_size()
    print(f"  {name:<22} {bytes_per_elem} 字节/元素")

# 类型转换
x = torch.randn(3)
print(f"\n原始类型: {x.dtype}")
print(f"转 float16: {x.half().dtype}")    # .half() = FP16
print(f"转 int: {x.int().dtype}")
print(f"转 bool: {(x > 0).dtype}")

print("\n提示: 训练时用 float32，推理时可用 float16 节省显存")

---
## 2. 形状变换：张量的「变形术」

神经网络中，数据需要不断变换形状才能流过不同的层。掌握 `reshape`、`view`、`transpose`、`squeeze` 等操作至关重要。

In [ ]:
# =============================================
# 2.1 reshape 和 view
# =============================================
# reshape 和 view 功能类似，都是改变形状但不改变数据
# view 要求内存连续，reshape 更灵活

x = torch.arange(12)  # [0, 1, 2, ..., 11]
print(f"原始张量: {x}")
print(f"形状: {x.shape}")

# 变成 3 行 4 列
a = x.reshape(3, 4)
print(f"\nreshape(3, 4):\n{a}")

# 变成 2 行 6 列
b = x.reshape(2, 6)
print(f"\nreshape(2, 6):\n{b}")

# 用 -1 自动推算某个维度
c = x.reshape(4, -1)  # 12 / 4 = 3，自动推出 3
print(f"\nreshape(4, -1) → {c.shape}:\n{c}")

# 变成 3 维: [batch=2, seq=3, features=2]
d = x.reshape(2, 3, 2)
print(f"\nreshape(2, 3, 2) → 3维张量:\n{d}")
print(f"形状: {d.shape} → [batch=2, seq_len=3, features=2]")

In [ ]:
# =============================================
# 2.2 squeeze 和 unsqueeze
# =============================================
# squeeze: 去掉大小为 1 的维度
# unsqueeze: 在指定位置添加大小为 1 的维度

x = torch.randn(3, 4)
print(f"原始: {x.shape}")

# 添加维度
y = x.unsqueeze(0)  # 在第 0 维添加 → [1, 3, 4]
print(f"unsqueeze(0): {y.shape}  ← 模拟 batch 维度")

z = x.unsqueeze(-1)  # 在最后添加 → [3, 4, 1]
print(f"unsqueeze(-1): {z.shape}")

# 去掉多余维度
w = y.squeeze(0)  # 去掉第 0 维的 1 → [3, 4]
print(f"squeeze(0): {w.shape}  ← 去掉 batch 维度")

print("\n常见场景:")
print("  - 单条数据预测时需要 unsqueeze(0) 添加 batch 维度")
print("  - 模型输出后用 squeeze() 去掉多余维度")

In [ ]:
# =============================================
# 2.3 转置和维度交换
# =============================================

# 2D 转置
x = torch.randn(3, 5)
print(f"原始: {x.shape}")
print(f"转置 (.T): {x.T.shape}")

# 高维度交换 (在注意力机制中很常见)
# 假设: [batch=2, heads=8, seq_len=10, head_dim=64]
attn = torch.randn(2, 8, 10, 64)
print(f"\n注意力张量: {attn.shape}")
print(f"  [batch=2, heads=8, seq_len=10, head_dim=64]")

# 交换 heads 和 seq_len: [batch, seq_len, heads, head_dim]
attn_t = attn.transpose(1, 2)
print(f"\ntranspose(1,2): {attn_t.shape}")
print(f"  [batch=2, seq_len=10, heads=8, head_dim=64]")

# permute 可以一次性重排所有维度
attn_p = attn.permute(0, 2, 1, 3)
print(f"\npermute(0,2,1,3): {attn_p.shape}  (与 transpose 结果相同)")

In [ ]:
# =============================================
# 2.4 拼接与拆分
# =============================================

a = torch.randn(2, 3)
b = torch.randn(2, 3)
print(f"a: {a.shape}")
print(f"b: {b.shape}")

# cat: 沿已有维度拼接
c0 = torch.cat([a, b], dim=0)  # 沿行方向拼接
c1 = torch.cat([a, b], dim=1)  # 沿列方向拼接
print(f"\ncat(dim=0): {c0.shape}  ← 行数翻倍")
print(f"cat(dim=1): {c1.shape}  ← 列数翻倍")

# stack: 沿新维度堆叠
s = torch.stack([a, b], dim=0)  # 添加新的第 0 维
print(f"\nstack(dim=0): {s.shape}  ← 新增 batch 维度")

# chunk: 均匀拆分
parts = torch.chunk(c0, 2, dim=0)  # 拆成 2 份
print(f"\nchunk 拆分: {[p.shape for p in parts]}")

print("\n常见场景:")
print("  - cat: 拼接多个序列、合并多头注意力输出")
print("  - stack: 将多个样本组成一个 batch")
print("  - chunk: 多头注意力中拆分 Q/K/V")

---
## 3. 张量数学运算

深度学习的核心就是大量的数学运算。PyTorch 张量支持丰富的数学运算，而且自动支持 GPU 加速。

In [ ]:
# =============================================
# 3.1 逐元素运算
# =============================================

a = torch.tensor([1.0, 2.0, 3.0, 4.0])
b = torch.tensor([2.0, 3.0, 4.0, 5.0])

print("逐元素运算（对应位置一一运算）:")
print(f"  a + b = {a + b}")     # 加法
print(f"  a * b = {a * b}")     # 乘法（不是矩阵乘法！）
print(f"  a / b = {a / b}")     # 除法
print(f"  a ** 2 = {a ** 2}")   # 幂运算
print(f"  sqrt(a) = {torch.sqrt(a)}")   # 开方
print(f"  exp(a) = {torch.exp(a)}")     # 指数
print(f"  log(a) = {torch.log(a)}")     # 自然对数

In [ ]:
# =============================================
# 3.2 矩阵乘法 — 神经网络最核心的运算
# =============================================
# 全连接层的本质: output = input @ weight + bias

# 定义输入和权重
x = torch.randn(4, 3)    # 4个样本，每个有3个特征
W = torch.randn(3, 5)    # 权重矩阵: 3输入 -> 5输出
b = torch.randn(5)       # 偏置

# 三种等价的矩阵乘法写法
out1 = x @ W              # Python 运算符（最简洁）
out2 = torch.matmul(x, W) # 函数形式（最通用）
out3 = x.mm(W)            # 方法形式（仅2D）

print(f"输入 x: {x.shape}  [batch=4, in_features=3]")
print(f"权重 W: {W.shape}  [in_features=3, out_features=5]")
print(f"输出:   {out1.shape}  [batch=4, out_features=5]")

# 加上偏置 — 这就是一个全连接层做的事情
output = x @ W + b  # 广播机制自动把 b 加到每一行
print(f"\n全连接层: output = x @ W + b")
print(f"输出形状: {output.shape}")

# 验证三种写法结果一致
assert torch.allclose(out1, out2) and torch.allclose(out2, out3)
print("三种写法结果一致!")

In [ ]:
# =============================================
# 3.3 广播机制（Broadcasting）
# =============================================
# 不同形状的张量也能做运算，PyTorch 会自动「扩展」较小的张量

# 矩阵 + 向量: 向量自动广播到每一行
matrix = torch.ones(3, 4)
row = torch.tensor([1.0, 2.0, 3.0, 4.0])  # 形状 [4]

result = matrix + row  # [3,4] + [4] → row 被广播成 [3,4]
print(f"矩阵 (3x4) + 行向量 (4,):")
print(result)

# 列向量广播
col = torch.tensor([[10.0], [20.0], [30.0]])  # 形状 [3,1]
result2 = matrix + col  # [3,4] + [3,1] → col 被广播成 [3,4]
print(f"\n矩阵 (3x4) + 列向量 (3,1):")
print(result2)

print("\n广播规则:")
print("  1. 从最后一个维度开始比较")
print("  2. 维度大小相同，或者其中一个为 1，就可以广播")
print("  3. 大小为 1 的维度会自动扩展")

In [ ]:
# =============================================
# 3.4 归约运算（聚合运算）
# =============================================
# 归约运算把多个值合并成一个（或沿某个维度合并）

x = torch.tensor([[1.0, 2.0, 3.0],
                   [4.0, 5.0, 6.0]])
print(f"x =\n{x}\n")

# 全局归约
print(f"sum: {x.sum()}")       # 所有元素之和
print(f"mean: {x.mean()}")     # 平均值
print(f"max: {x.max()}")       # 最大值
print(f"min: {x.min()}")       # 最小值

# 沿指定维度归约
print(f"\n沿 dim=0 求和 (每列之和): {x.sum(dim=0)}")  # [5, 7, 9]
print(f"沿 dim=1 求和 (每行之和): {x.sum(dim=1)}")  # [6, 15]
print(f"沿 dim=1 求均值 (每行均值): {x.mean(dim=1)}")  # [2, 5]

# argmax: 返回最大值的索引，在分类问题中很常用
logits = torch.tensor([0.2, 0.5, 0.1, 0.8, 0.3])  # 模型输出的分数
predicted_class = logits.argmax()
print(f"\nlogits: {logits}")
print(f"预测类别 (argmax): {predicted_class}  (第 {predicted_class} 个类别得分最高)")

---
## 4. 神经网络的本质：函数拟合

### 一个改变认知的想法

先忘掉"人工智能"这四个字。神经网络的本质非常简单：

> **神经网络 = 一个超级复杂的函数**
>
> 输入 → `f(x)` → 输出

就像你写过的任何函数一样。区别在于：你写的函数规则是手动编写的；神经网络的规则是从数据中自动"学"出来的。

### 函数拟合的直觉

```
问题：给你一堆数据点，找到一个函数来描述它们

数据：x = [1, 2, 3, 4, 5],  y = [2.1, 3.9, 6.2, 7.8, 10.1]

人类的判断：这大概是 y = 2x 的关系

神经网络的做法：
  1. 先随机猜一个函数：y = 0.5x + 3  （猜错了）
  2. 用数据检验，发现差距很大
  3. 调整参数：y = 1.2x + 1.5  （好一些了）
  4. 继续调整：y = 1.8x + 0.3  （更好了）
  5. 最终收敛：y = 2.0x + 0.1  （非常接近了！）

这个不断调整的过程，就是"训练"。
```

### 最简单的神经网络：线性回归

线性回归就是一条直线：**y = wx + b**

- **w** (weight, 权重) = 直线的斜率
- **b** (bias, 偏置) = 直线的截距
- 模型要学习的就是 w 和 b 的值！
- 训练前：w=0.5, b=3（随机猜的）→ 训练后：w=2.0, b=0.1（从数据中学到的）

### 从线性到非线性：激活函数

![常用激活函数对比](images/activation_functions.png)

*图2：常用激活函数对比*

现实世界的数据很少是直线关系。解决方案：在线性运算后加一个"弯曲"的函数。

```
输入 → [线性运算: wx+b] → [激活函数: 弯一下] → 输出

常用的激活函数：

  ReLU（最常用）：f(x) = max(0, x)
  负数变成0，正数不变。像一个单向阀门，只让正数通过

  GELU（大模型常用）：f(x) = x * Φ(x)
  比 ReLU 更平滑，GPT/BERT 都用这个
```

### 多层神经网络

把多个"线性+激活"堆叠起来，就成了深度神经网络：

```
输入层        隐藏层1       隐藏层2       输出层
(Input)      (Hidden 1)    (Hidden 2)    (Output)

 x1 ──┐
       ├──→ [wx+b] → [ReLU] ──┐
 x2 ──┤                       ├──→ [wx+b] → [ReLU] ──→ [wx+b] → y
       ├──→ [wx+b] → [ReLU] ──┤
 x3 ──┘                       └──→ ...

每一层做的事情：
  1. 线性变换：output = input x W + b  （矩阵乘法！）
  2. 激活函数：output = ReLU(output)    （加入非线性）

层数越多，函数就能拟合越复杂的模式
这就是"深度学习"中"深度"的含义！
```

**类比：流水线加工** — 每一层都在对数据做一次"变换"，就像工厂流水线中每个工序都在加工产品。原材料经过多道工序，最终变成成品。

> 💡 **记忆要点**
> - 神经网络 = 一个从数据中自动学习规则的函数
> - 核心操作就两步：**线性变换**（矩阵乘法）+ **激活函数**（加入非线性）
> - "训练"就是不断调整参数（w 和 b）使函数的输出接近正确答案
> - 层数越深，能拟合的函数越复杂，但也越难训练

---
## 5. 前向传播与损失函数

### 什么是前向传播？

前向传播就是数据从输入层流向输出层的过程，非常直观：

```
前向传播 = 把数据从左往右"传"一遍

  输入          第1层           第2层          输出
  [x1]    →   [h1 = w1*x+b1]  →  [h2 = w2*h1+b2]  →  [预测值 y_hat]
  [x2]    →   [    ReLU      ]  →  [     ReLU      ]  →
  [x3]    →                    →                    →
```

![前向传播流程](images/forward_propagation.png)

*图3：神经网络前向传播*

**类比：工厂流水线**
- 原材料（输入数据）→ 第1道工序（第1层变换）→ 第2道工序（第2层变换）→ 成品（预测结果）
- 每一步的输出是下一步的输入
- 给定相同的输入，输出完全确定

### 损失函数：衡量"错"了多少

前向传播给出了一个预测值，但我们需要知道"预测得准不准"。损失函数就是衡量预测值和真实值之间差距的工具。

**损失越小 = 预测越准 = 模型越好。训练的目标就是让损失不断减小！**

**常见的损失函数**：

| 损失函数 | 公式 | 用途 |
|---------|------|------|
| MSE（均方误差） | `(预测值 - 真实值)²` | 回归任务 |
| 交叉熵 | `-log(预测正确类的概率)` | 分类/语言模型 |

交叉熵的直觉：

| 预测正确类的概率 | Loss = -log(p) | 评价 |
|:---:|:---:|:---:|
| 1.0 | 0.0 | 完美预测 |
| 0.9 | 0.1 | 很好 |
| 0.5 | 0.69 | 一般 |
| 0.1 | 2.3 | 很差 |
| 0.01 | 4.6 | 糟糕 |

概率越低 → 损失越大 → 需要更大幅度的调整。

> 💡 **记忆要点**
> - 前向传播 = 数据从输入到输出的**单方向流动**
> - 每一层做的事：**矩阵乘法 + 加偏置 + 激活函数**
> - 损失函数衡量预测值和真实值的差距，**损失越小 = 模型越好**
> - 回归任务用 **MSE**，分类/语言模型用**交叉熵**

---
## 6. 反向传播与梯度下降

### 核心问题：参数该怎么调？

我们知道了前向传播算预测值，损失函数衡量差距。现在的问题是：模型有成千上万个参数，每个参数应该增大还是减小？增大/减小多少？

答案就是：**反向传播（Backpropagation）**。它告诉每个参数："你应该增大 0.03" / "你应该减小 0.15" / "你几乎不用变"。

### 梯度：调整的方向和幅度

反向传播的核心概念是"梯度"（Gradient）：

> **梯度 = 如果这个参数稍微增大一点，损失会怎么变**

- 梯度为正 → 增大参数会让损失增大 → 应该**减小**参数
- 梯度为负 → 增大参数会让损失减小 → 应该**增大**参数
- 梯度接近 0 → 参数已经差不多了

**直觉理解**：想象你蒙着眼睛站在山坡上，要走到山谷最低点。你的策略：1) 用脚试探周围的坡度（计算梯度）；2) 往最陡的下坡方向走一步（梯度下降）；3) 重复，直到走到谷底（损失最小）。

### 反向传播的流动方向

![反向传播梯度流](images/backpropagation.png)

*图4：反向传播流程*

```
前向传播：数据从左到右
  输入 ──→ 第1层 ──→ 第2层 ──→ 输出 ──→ 损失

反向传播：梯度从右到左（方向完全相反！）
  输入 ←── 第1层 ←── 第2层 ←── 输出 ←── 损失
```

PyTorch 的强大之处：自动帮你算所有梯度！

```python
# 你只需要写前向传播
y = model(x)
loss = criterion(y, target)

# 一行代码完成反向传播
loss.backward()  # PyTorch 自动计算所有梯度！
```

### 参数更新公式

梯度下降的更新公式非常简单：

> **新参数 = 旧参数 - 学习率 x 梯度**
>
> `w_new = w_old - lr * grad_w`

就是这么简单！整个深度学习的核心就是这一行公式。

### 学习率的重要性

![损失下降与梯度下降优化](images/loss_descent.png)

*图5：损失下降曲线与梯度下降优化*

| 学习率 | 效果 | 类比 |
|--------|------|------|
| 太大 (lr=1.0) | 来回跳，找不到最低点 | 步子太大，跨过了山谷 |
| 太小 (lr=0.0001) | 龟速前进，训练很慢 | 每步只挪一毫米 |
| 刚好 (lr=0.01) | 稳步到达最低点 | 步幅合适，稳步下山 |

常用学习率范围：

| 任务 | 常用学习率 |
|------|-----------|
| 预训练大模型 | 1e-4 到 3e-4 |
| 微调大模型 | 1e-5 到 5e-5 |
| 简单线性回归 | 0.01 到 0.1 |

### 完整的训练循环

把所有步骤串起来，就是一个完整的训练循环——**五步法：前向 → 损失 → 反向 → 更新 → 清零**：

```python
for epoch in range(100):        # 训练 100 轮
    for batch in dataloader:    # 每轮遍历所有数据

        # 1. 前向传播
        y_pred = model(x)       # 计算预测值

        # 2. 计算损失
        loss = MSE(y_pred, y)   # 预测 vs 真实

        # 3. 反向传播
        loss.backward()         # 计算所有梯度

        # 4. 更新参数
        optimizer.step()        # w = w - lr * grad

        # 5. 清空梯度
        optimizer.zero_grad()   # 准备下一轮
```

### Epoch、Batch、Iteration 的区别

假设你有 1000 条训练数据，batch_size = 100：

| 概念 | 含义 | 本例中的值 | 类比 |
|------|------|-----------|------|
| **Epoch** | 全部数据过一遍 | 1 epoch = 看完 1000 条 | 把教科书从头读一遍 |
| **Batch** | 每次取一小批数据 | 1000 / 100 = 10 个 batch | 每次只读一个章节 |
| **Iteration** | 处理 1 个 batch | 1 epoch = 10 iterations | 读完一章并做笔记 |

所以：50 epochs x 10 batches/epoch = 500 iterations，模型参数被更新了 500 次。

> 💡 **记忆要点**
> - 反向传播 = 从损失出发，反向计算每个参数的梯度
> - 参数更新公式：**w = w - lr * gradient**，这是深度学习最核心的公式
> - 学习率控制步长：太大会震荡，太小会太慢
> - 训练循环五步法：**前向 → 损失 → 反向 → 更新 → 清零**
> - PyTorch 用 `loss.backward()` 自动完成所有梯度计算

---
## 7. 自动求导（Autograd）— 反向传播的基石

PyTorch 的 `autograd` 系统自动完成梯度计算。你只需要设置 `requires_grad=True`，然后调用 `.backward()`，所有梯度就自动算好了。

In [ ]:
# =============================================
# 4.1 基本梯度计算
# =============================================

# 例1: y = x^2，dy/dx = 2x
x = torch.tensor(3.0, requires_grad=True)  # 告诉 PyTorch: 我需要对 x 求导
y = x ** 2
y.backward()  # 反向传播，计算梯度

print("例1: y = x^2")
print(f"  x = {x.item()}")
print(f"  y = x^2 = {y.item()}")
print(f"  dy/dx = 2x = {x.grad.item()}  (期望值: {2 * x.item()})")

# 例2: z = (x + y)^2，对 x 和 y 分别求导
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)
z = (x + y) ** 2  # z = (2+3)^2 = 25
z.backward()

print("\n例2: z = (x + y)^2")
print(f"  x = {x.item()}, y = {y.item()}")
print(f"  z = (x+y)^2 = {z.item()}")
print(f"  dz/dx = 2(x+y) = {x.grad.item()}  (期望值: {2 * (2 + 3)})")
print(f"  dz/dy = 2(x+y) = {y.grad.item()}  (期望值: {2 * (2 + 3)})")

In [ ]:
# =============================================
# 4.2 计算图的可视化理解
# =============================================
# PyTorch 在运算过程中自动构建「计算图」
# 反向传播就是沿着计算图，用链式法则逐步求导

# 模拟神经网络中的一次计算
# 前向传播: input → 乘权重 → 加偏置 → 激活函数 → 输出
x = torch.tensor([1.0, 2.0, 3.0])                    # 输入（不需要梯度）
w = torch.tensor([0.5, -0.3, 0.8], requires_grad=True)  # 权重（需要梯度）
b = torch.tensor(0.1, requires_grad=True)               # 偏置（需要梯度）

# 前向传播过程
z = (x * w).sum() + b       # 线性变换: z = x·w + b
a = torch.sigmoid(z)        # 激活函数
loss = (a - 1.0) ** 2       # 假设目标值是 1.0，用 MSE 损失

print("前向传播:")
print(f"  输入 x = {x}")
print(f"  权重 w = {w.data}")
print(f"  偏置 b = {b.data.item():.1f}")
print(f"  线性输出 z = x·w + b = {z.item():.4f}")
print(f"  激活输出 a = sigmoid(z) = {a.item():.4f}")
print(f"  损失 loss = (a - 1)^2 = {loss.item():.4f}")

# 反向传播
loss.backward()

print("\n反向传播 (梯度):")
print(f"  d(loss)/dw = {w.grad}")
print(f"  d(loss)/db = {b.grad.item():.4f}")
print("\n计算图: x*w → sum → +b → sigmoid → (a-1)^2")
print("链式法则: 梯度从 loss 沿箭头反方向一路传回 w 和 b")

In [ ]:
# =============================================
# 4.3 梯度累积与清零
# =============================================
# 重要: PyTorch 的梯度是 **累加** 的，不是覆盖!
# 每次 backward() 前必须清零，否则梯度会越来越大

w = torch.tensor(2.0, requires_grad=True)

# 第一次 backward
y1 = w * 3
y1.backward()
print(f"第1次 backward 后 w.grad = {w.grad.item()}")  # 3.0

# 第二次 backward（不清零）
y2 = w * 3
y2.backward()
print(f"第2次 backward 后 w.grad = {w.grad.item()}")  # 6.0 (3+3，累加了!)

# 正确做法：每次 backward 前清零
w.grad.zero_()  # 原地清零
y3 = w * 3
y3.backward()
print(f"清零后 backward: w.grad = {w.grad.item()}")   # 3.0 (正确!)

print("\n关键规则: optimizer.zero_grad() 必须在 loss.backward() 之前调用")
print("忘记清零是初学者最常见的 bug 之一!")

In [ ]:
# =============================================
# 4.4 torch.no_grad() — 推理时关闭梯度
# =============================================
# 推理（预测）时不需要计算梯度，关闭它可以节省内存和加速

x = torch.randn(1000, 1000, requires_grad=True)

# 有梯度时
y = x * 2 + 1
print(f"有梯度:   y.requires_grad = {y.requires_grad}")

# 无梯度时（推理模式）
with torch.no_grad():
    y_no_grad = x * 2 + 1
    print(f"无梯度:   y.requires_grad = {y_no_grad.requires_grad}")

# .detach() 也能切断梯度（用于取出中间结果做可视化等）
y_detach = y.detach()
print(f"detach(): y.requires_grad = {y_detach.requires_grad}")

print("\n常见用法:")
print("  - model.eval() + torch.no_grad() → 推理时使用")
print("  - .detach() → 从计算图中取出张量做可视化")
print("  - .item() → 取出标量值做日志记录")

---
## 8. 从零实现梯度下降

在使用 `nn.Module` 之前，让我们先手动实现前向传播和反向传播，深入理解神经网络"学习"的本质。

**目标**：用一个单神经元学习 `y = 3x + 2`

In [ ]:
# =============================================
# 5.1 手动实现梯度下降
# =============================================

# 目标: 学习 y = 3x + 2
# 模型: y_pred = w * x + b
# 任务: 让模型自动找到 w=3, b=2

# 生成训练数据
torch.manual_seed(42)
X = torch.linspace(-5, 5, 100)  # 100个等间距的 x 值
Y = 3 * X + 2 + 0.5 * torch.randn(100)  # y = 3x + 2 + 噪声

# 初始化参数（随机值）
w = torch.tensor(0.0, requires_grad=True)  # 权重，初始化为 0
b = torch.tensor(0.0, requires_grad=True)  # 偏置，初始化为 0

learning_rate = 0.01
n_epochs = 100
history = []  # 记录训练过程

print(f"初始参数: w = {w.item():.4f}, b = {b.item():.4f}")
print(f"目标参数: w = 3.0000, b = 2.0000\n")

for epoch in range(n_epochs):
    # --- 前向传播 ---
    Y_pred = w * X + b           # 用当前参数做预测
    loss = ((Y_pred - Y) ** 2).mean()  # MSE 损失
    
    # --- 反向传播 ---
    loss.backward()  # 自动计算 d(loss)/dw 和 d(loss)/db
    
    # --- 参数更新 ---
    # with torch.no_grad() 因为参数更新不需要被追踪
    with torch.no_grad():
        w -= learning_rate * w.grad  # w = w - lr * gradient
        b -= learning_rate * b.grad  # b = b - lr * gradient
    
    # 记录历史
    history.append({
        'epoch': epoch,
        'loss': loss.item(),
        'w': w.item(),
        'b': b.item()
    })
    
    # --- 清零梯度（关键!）---
    w.grad.zero_()
    b.grad.zero_()
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:>3}  Loss: {loss.item():.4f}  "
              f"w: {w.item():.4f}  b: {b.item():.4f}")

print(f"\n最终参数: w = {w.item():.4f} (目标 3.0), b = {b.item():.4f} (目标 2.0)")
print(f"参数误差: w 误差 {abs(w.item() - 3.0):.4f}, b 误差 {abs(b.item() - 2.0):.4f}")

In [ ]:
# =============================================
# 5.2 可视化训练过程
# =============================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- 图1: 损失下降曲线 ---
losses = [h['loss'] for h in history]
axes[0].plot(losses, color='steelblue', linewidth=2)
axes[0].set_title('损失曲线', fontsize=14)
axes[0].set_xlabel('训练轮次')
axes[0].set_ylabel('MSE 损失')
axes[0].set_yscale('log')  # 对数坐标更直观
axes[0].grid(True, alpha=0.3)

# --- 图2: 参数收敛过程 ---
ws = [h['w'] for h in history]
bs = [h['b'] for h in history]
axes[1].plot(ws, label='w (target=3.0)', color='crimson', linewidth=2)
axes[1].plot(bs, label='b (target=2.0)', color='forestgreen', linewidth=2)
axes[1].axhline(y=3.0, color='crimson', linestyle='--', alpha=0.5)
axes[1].axhline(y=2.0, color='forestgreen', linestyle='--', alpha=0.5)
axes[1].set_title('参数收敛过程', fontsize=14)
axes[1].set_xlabel('训练轮次')
axes[1].set_ylabel('参数值')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# --- 图3: 模型拟合效果 ---
axes[2].scatter(X.numpy(), Y.numpy(), alpha=0.4, s=15, color='gray', label='数据点')
# 初始模型
x_plot = torch.linspace(-5, 5, 100)
y_init = history[0]['w'] * x_plot + history[0]['b']
axes[2].plot(x_plot.numpy(), y_init.numpy(), color='orange',
             linewidth=2, linestyle='--', label=f'Epoch 1 (w={history[0]["w"]:.1f})')
# 最终模型
y_final = w.item() * x_plot + b.item()
axes[2].plot(x_plot.numpy(), y_final.numpy(), color='red',
             linewidth=2, label=f'Final (w={w.item():.2f}, b={b.item():.2f})')
# 真实函数
y_true = 3 * x_plot + 2
axes[2].plot(x_plot.numpy(), y_true.numpy(), color='blue',
             linewidth=2, linestyle=':', label='True: y=3x+2')
axes[2].set_title('模型拟合效果', fontsize=14)
axes[2].set_xlabel('x')
axes[2].set_ylabel('y')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("左图: 损失值在训练过程中快速下降")
print("中图: 参数 w 和 b 逐渐收敛到目标值")
print("右图: 模型从一条水平线（初始）变成了正确的拟合线")

---
## 9. 用 PyTorch nn.Module 实现线性回归

上面我们手动实现了梯度下降，理解了底层原理。现在来学 PyTorch 的"正规军"写法 — 用 `nn.Module` 和 `Optimizer`。这种写法是后续所有章节的标准模式。

In [ ]:
# =============================================
# 6.1 定义模型
# =============================================
# nn.Module 是 PyTorch 中所有神经网络的基类
# 继承它，实现 __init__（定义层）和 forward（定义计算流程）

class LinearRegression(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()  # 必须调用父类的初始化
        # nn.Linear 自动创建 weight 和 bias 参数
        self.linear = nn.Linear(input_dim, output_dim)
    
    def forward(self, x):
        # 前向传播: 定义数据如何流过模型
        return self.linear(x)


# 创建模型
model = LinearRegression(1, 1)  # 1个输入特征，1个输出

# 查看模型结构
print("模型结构:")
print(model)

# 查看参数
print("\n模型参数:")
for name, param in model.named_parameters():
    print(f"  {name}: shape={param.shape}, value={param.data}")

total_params = sum(p.numel() for p in model.parameters())
print(f"\n总参数量: {total_params}")

In [ ]:
# =============================================
# 6.2 标准训练流程
# =============================================

# --- 准备数据 ---
torch.manual_seed(42)
X_train = torch.linspace(-5, 5, 200).unsqueeze(1)   # [200, 1]
Y_train = 3 * X_train + 2 + 0.5 * torch.randn_like(X_train)  # y = 3x + 2 + 噪声

# --- 创建模型、损失函数、优化器 ---
model = LinearRegression(1, 1)
criterion = nn.MSELoss()              # 损失函数: 均方误差
optimizer = torch.optim.SGD(           # 优化器: 随机梯度下降
    model.parameters(),
    lr=0.01
)

print("开始训练...")
print(f"{'Epoch':>6} {'Loss':>12} {'Weight':>10} {'Bias':>10}")
print("-" * 42)

losses = []

for epoch in range(100):
    # 1. 前向传播: 模型预测
    Y_pred = model(X_train)
    
    # 2. 计算损失
    loss = criterion(Y_pred, Y_train)
    
    # 3. 反向传播: 计算梯度
    optimizer.zero_grad()  # 清零梯度
    loss.backward()        # 反向传播
    
    # 4. 更新参数
    optimizer.step()       # 根据梯度更新权重
    
    losses.append(loss.item())
    
    if (epoch + 1) % 20 == 0:
        w_val = model.linear.weight.item()
        b_val = model.linear.bias.item()
        print(f"{epoch+1:>6} {loss.item():>12.4f} {w_val:>10.4f} {b_val:>10.4f}")

# 最终结果
w_final = model.linear.weight.item()
b_final = model.linear.bias.item()
print(f"\n学到的参数: y = {w_final:.4f}x + {b_final:.4f}")
print(f"目标函数:   y = 3.0000x + 2.0000")

In [ ]:
# =============================================
# 6.3 可视化: 损失曲线 + 拟合效果
# =============================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图: 损失曲线
axes[0].plot(losses, color='steelblue', linewidth=2)
axes[0].set_title('训练损失 (nn.Module)', fontsize=14)
axes[0].set_xlabel('训练轮次')
axes[0].set_ylabel('MSE 损失')
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3)

# 右图: 拟合效果
model.eval()
with torch.no_grad():
    Y_pred = model(X_train)

axes[1].scatter(X_train.numpy(), Y_train.numpy(),
                alpha=0.3, s=10, color='gray', label='训练数据')
axes[1].plot(X_train.numpy(), Y_pred.numpy(),
             color='red', linewidth=2, label=f'Model: y={w_final:.2f}x+{b_final:.2f}')
axes[1].plot(X_train.numpy(), (3 * X_train + 2).numpy(),
             color='blue', linewidth=2, linestyle='--', label='真实值: y=3x+2')
axes[1].set_title('模型预测 vs 真实值', fontsize=14)
axes[1].set_xlabel('x')
axes[1].set_ylabel('y')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 10. 多层神经网络：从线性到非线性

线性回归只能学线性关系。真实世界的数据往往是非线性的。通过**堆叠多个线性层 + 激活函数**，神经网络可以拟合任意复杂的函数。

**激活函数的作用**：如果没有激活函数，多层线性层等价于一层。激活函数引入非线性，让网络有能力学习曲线。

In [ ]:
# =============================================
# 7.1 常见激活函数
# =============================================

x = torch.linspace(-5, 5, 200)

activations = {
    'ReLU': torch.relu(x),
    'Sigmoid': torch.sigmoid(x),
    'Tanh': torch.tanh(x),
    'GELU': nn.functional.gelu(x),  # Transformer 中最常用
}

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, (name, y) in zip(axes, activations.items()):
    ax.plot(x.numpy(), y.numpy(), linewidth=2, color='steelblue')
    ax.axhline(y=0, color='gray', linewidth=0.5)
    ax.axvline(x=0, color='gray', linewidth=0.5)
    ax.set_title(name, fontsize=14)
    ax.set_xlabel('x')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("各激活函数的特点:")
print("  ReLU:    简单高效，max(0,x)。大部分网络的默认选择")
print("  Sigmoid: 输出在 (0,1)，用于二分类输出层")
print("  Tanh:    输出在 (-1,1)，比 Sigmoid 更常用")
print("  GELU:    平滑版 ReLU，GPT/BERT 等 Transformer 模型的标配")

In [ ]:
# =============================================
# 7.2 拟合非线性函数: y = sin(x)
# =============================================

# 定义一个多层神经网络
class MLP(nn.Module):
    """多层感知机 (Multi-Layer Perceptron)"""
    def __init__(self, hidden_size=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, hidden_size),    # 输入层
            nn.ReLU(),                    # 激活函数
            nn.Linear(hidden_size, hidden_size),  # 隐藏层
            nn.ReLU(),
            nn.Linear(hidden_size, 1),    # 输出层
        )
    
    def forward(self, x):
        return self.net(x)


# --- 准备非线性数据 ---
torch.manual_seed(42)
X = torch.linspace(-2 * np.pi, 2 * np.pi, 500).unsqueeze(1)
Y = torch.sin(X) + 0.1 * torch.randn_like(X)  # y = sin(x) + 噪声

# --- 创建模型 ---
model = MLP(hidden_size=64)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)  # Adam 优化器更快收敛

param_count = sum(p.numel() for p in model.parameters())
print(f"模型参数量: {param_count}")
print(f"模型结构: {model}")

# --- 训练 ---
losses = []
snapshots = {}  # 保存中间预测结果

for epoch in range(1000):
    Y_pred = model(X)
    loss = criterion(Y_pred, Y)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    
    # 保存快照用于可视化
    if epoch in [0, 10, 50, 200, 999]:
        model.eval()
        with torch.no_grad():
            snapshots[epoch] = model(X).numpy().flatten()
        model.train()
    
    if (epoch + 1) % 200 == 0:
        print(f"Epoch {epoch+1:>5}  Loss: {loss.item():.6f}")

print(f"\n最终损失: {losses[-1]:.6f}")

In [ ]:
# =============================================
# 7.3 可视化: 模型逐步学会拟合 sin(x)
# =============================================

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 左图: 损失曲线
axes[0].plot(losses, color='steelblue', linewidth=1.5)
axes[0].set_title('损失曲线 (MLP 学习 sin(x))', fontsize=14)
axes[0].set_xlabel('训练轮次')
axes[0].set_ylabel('MSE 损失')
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3)

# 右图: 拟合过程
X_np = X.numpy().flatten()
axes[1].scatter(X_np, Y.numpy().flatten(), alpha=0.15, s=5, color='gray', label='数据点')

colors = ['orange', 'gold', 'limegreen', 'dodgerblue', 'red']
for (ep, pred), color in zip(snapshots.items(), colors):
    label = f'Epoch {ep+1}'
    alpha = 0.4 if ep < 999 else 1.0
    lw = 1.5 if ep < 999 else 2.5
    axes[1].plot(X_np, pred, color=color, linewidth=lw, alpha=alpha, label=label)

axes[1].plot(X_np, np.sin(X_np), color='blue', linewidth=2,
             linestyle=':', label='真实值: sin(x)')
axes[1].set_title('MLP 学习拟合 sin(x)', fontsize=14)
axes[1].set_xlabel('x')
axes[1].set_ylabel('y')
axes[1].legend(loc='upper right', fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("关键观察:")
print("  - Epoch 1: 模型输出几乎是一条直线（随机初始化）")
print("  - Epoch 50: 开始捕捉到大致的波形")
print("  - Epoch 1000: 几乎完美拟合了 sin(x)")
print("  - 这就是'万能近似定理': 足够宽的 MLP 可以拟合任意连续函数")

---
## 11. 学习率的影响

学习率是最重要的超参数之一。让我们直观地对比不同学习率的效果。

In [ ]:
# =============================================
# 8.1 对比不同学习率
# =============================================

# 准备数据
torch.manual_seed(42)
X_lr = torch.linspace(-3, 3, 100).unsqueeze(1)
Y_lr = 2 * X_lr + 1 + 0.3 * torch.randn_like(X_lr)

learning_rates = [0.5, 0.05, 0.001]
lr_histories = {}

for lr in learning_rates:
    torch.manual_seed(42)  # 相同初始化
    model_lr = LinearRegression(1, 1)
    optimizer_lr = torch.optim.SGD(model_lr.parameters(), lr=lr)
    criterion_lr = nn.MSELoss()
    
    losses_lr = []
    for epoch in range(100):
        pred = model_lr(X_lr)
        loss = criterion_lr(pred, Y_lr)
        optimizer_lr.zero_grad()
        loss.backward()
        optimizer_lr.step()
        # 如果损失爆炸则截断记录
        losses_lr.append(min(loss.item(), 1e4))
    
    lr_histories[lr] = losses_lr
    print(f"lr={lr:<6} 最终损失: {losses_lr[-1]:.4f}")

# 可视化
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['red', 'steelblue', 'orange']
for (lr, losses_lr), color in zip(lr_histories.items(), colors):
    ax.plot(losses_lr, label=f'lr={lr}', linewidth=2, color=color)

ax.set_title('学习率对训练的影响', fontsize=14)
ax.set_xlabel('训练轮次')
ax.set_ylabel('MSE 损失')
ax.set_yscale('log')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n观察:")
print(f"  lr=0.5:   学习率过大 — 损失可能震荡或发散")
print(f"  lr=0.05:  学习率适中 — 损失平稳下降，收敛较快")
print(f"  lr=0.001: 学习率过小 — 收敛太慢，100个 epoch 后损失仍然较高")

---
## 12. 损失函数景观的可视化

最后，让我们可视化损失函数在参数空间中的"地形图"。梯度下降就像一个小球在这个地形上滚动，寻找最低点。

In [ ]:
# =============================================
# 9.1 损失函数的 3D 景观
# =============================================

# 对于线性回归 y = wx + b，损失函数 L(w, b) 是一个碗状曲面
# 目标: 找到碗底的 (w, b)

# 用之前的数据
torch.manual_seed(42)
X_vis = torch.linspace(-3, 3, 50)
Y_vis = 3 * X_vis + 2 + 0.5 * torch.randn(50)

# 计算不同 (w, b) 下的 MSE 损失
w_range = np.linspace(-1, 7, 80)
b_range = np.linspace(-3, 7, 80)
W_grid, B_grid = np.meshgrid(w_range, b_range)
Loss_grid = np.zeros_like(W_grid)

for i in range(len(b_range)):
    for j in range(len(w_range)):
        w_val = w_range[j]
        b_val = b_range[i]
        y_pred = w_val * X_vis.numpy() + b_val
        Loss_grid[i, j] = np.mean((y_pred - Y_vis.numpy()) ** 2)

# 绘制等高线图 + 梯度下降路径
fig, ax = plt.subplots(figsize=(10, 8))

# 等高线
contour = ax.contourf(W_grid, B_grid, Loss_grid, levels=30, cmap='YlOrRd')
plt.colorbar(contour, label='MSE 损失')
ax.contour(W_grid, B_grid, Loss_grid, levels=15, colors='white', alpha=0.3, linewidths=0.5)

# 重新跑一次梯度下降，记录路径
torch.manual_seed(42)
w_path = torch.tensor(0.0, requires_grad=True)
b_path = torch.tensor(0.0, requires_grad=True)
path_w, path_b = [w_path.item()], [b_path.item()]

for _ in range(100):
    y_pred = w_path * X_vis + b_path
    loss = ((y_pred - Y_vis) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        w_path -= 0.01 * w_path.grad
        b_path -= 0.01 * b_path.grad
    w_path.grad.zero_()
    b_path.grad.zero_()
    path_w.append(w_path.item())
    path_b.append(b_path.item())

# 画路径
ax.plot(path_w, path_b, 'o-', color='cyan', markersize=2, linewidth=1.5,
        label='梯度下降轨迹')
ax.plot(path_w[0], path_b[0], 's', color='lime', markersize=12, label='起点')
ax.plot(path_w[-1], path_b[-1], '*', color='white', markersize=15, label='终点')
ax.plot(3, 2, 'x', color='yellow', markersize=15, markeredgewidth=3, label='目标 (w=3, b=2)')

ax.set_title('损失函数地形图 & 梯度下降轨迹', fontsize=14)
ax.set_xlabel('w（权重）', fontsize=12)
ax.set_ylabel('b（偏置）', fontsize=12)
ax.legend(loc='upper right')

plt.tight_layout()
plt.show()

print("理解这张图:")
print("  - 颜色表示损失值: 红色=高损失，浅色=低损失")
print("  - 碗底 (w=3, b=2) 是损失最小的点")
print("  - 青色路径是梯度下降的轨迹: 从起点沿最陡方向走向碗底")
print("  - 这就是神经网络'学习'的本质: 在参数空间中寻找损失最小的点")

---
## 13. 本章总结

**1. 张量** — 标量(0维) → 向量(1维) → 矩阵(2维) → 高维张量。shape 是最重要的属性。大模型输入 = `(batch, seq_len, hidden_dim)`

**2. 神经网络** — 本质 = 从数据中学习规则的函数。核心操作 = 矩阵乘法 + 激活函数。层数越多，能拟合越复杂的模式

**3. 前向传播** — 数据从输入到输出的单方向流动。每层：线性变换 → 激活函数

**4. 损失函数** — 衡量预测和真实值的差距。回归用 MSE，分类/语言模型用交叉熵

**5. 反向传播** — 从损失反向计算每个参数的梯度。PyTorch 自动完成：`loss.backward()`

**6. 梯度下降** — `w = w - lr * gradient`。训练五步法：前向→损失→反向→更新→清零

**核心公式**：
- 前向：`y = Wx + b`
- 损失：`L = (y - y_hat)²`
- 更新：`w = w - lr * dL/dw`

### 下一章预告

在第4章中，我们将学习**词嵌入（Embedding）的奥秘**：
- 为什么需要把文字变成数字
- One-hot vs Word2Vec vs Embedding
- 词向量的几何意义
- 找出"国王 - 男人 + 女人 ≈ 女王"

---

> **参考资料**
> - PyTorch Official Tutorial: Tensors — 张量基础操作
> - 3Blue1Brown: Neural Networks — 神经网络可视化讲解
> - Andrej Karpathy: Micrograd — 从零实现自动微分
> - *Deep Learning* (Goodfellow et al., 2016) Chapter 6 — 前馈网络